# 02 -- Training (dua fase, resume-safe)

Melatih satu backbone ConvNeXt per run notebook ini (ganti `BACKBONE` di
sel konfigurasi, jalankan lagi untuk backbone lain -- BUKAN 4 notebook
terpisah seperti eksperimen lama, supaya tidak ada copy-paste bug lagi).

**Prasyarat (wajib sebelum jalan):**
1. `results/split_manifest.json` sudah ada (dari `01_dataset_audit.ipynb`
   + resplit -- gerbang audit GAGAL 34.84% kebocoran, split lama dari
   Kaggle TIDAK dipakai lagi).
2. Dataset ada di `/content/drive/MyDrive/Bone_Fracture_Dataset` (struktur
   sama seperti lokal -- manifest pakai path RELATIF, jadi portable).
3. Repo `project-fracture` accessible dari Colab (public di GitHub) --
   sel berikutnya clone otomatis.

**Resume:** kalau sesi Colab terputus, jalankan ulang notebook ini dari
atas dengan `BACKBONE` yang sama -- `src/fracture/train.py` baca
`status.json` di `runs/<run_id>/` dan lanjut dari epoch terakhir yang
benar-benar tersimpan, TIDAK mengulang dari ImageNet weights.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Konfigurasi -- SATU-SATUNYA baris yang perlu diganti per run

In [ ]:
BACKBONE = "small"  # ganti ke "tiny" / "small" / "base" / "large"

DATASET_ROOT = "/content/drive/MyDrive/Bone_Fracture_Dataset"
REPO_URL = "https://github.com/alianhar/project-fracture.git"
REPO_DIR = "/content/project-fracture"
RUNS_ROOT = "/content/drive/MyDrive/fracture-runs"  # di Drive -- selamat dari disconnect

In [ ]:
import os

if not os.path.exists(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

import sys
sys.path.insert(0, REPO_DIR)

In [ ]:
!pip -q install pyyaml

In [ ]:
import json
import hashlib
from pathlib import Path

import yaml
import tensorflow as tf

from src.fracture.data import make_generators, compute_class_weight
from src.fracture.train import run_training

# Config terkunci + override backbone (spec Sec6.1) -- HANYA backbone yang
# beda antar model, seluruh hyperparameter lain identik.
with open(f"{REPO_DIR}/configs/base.yaml") as f:
    config = yaml.safe_load(f)

override_file = "base_model.yaml" if BACKBONE == "base" else f"{BACKBONE}.yaml"
with open(f"{REPO_DIR}/configs/{override_file}") as f:
    config.update(yaml.safe_load(f))

print(json.dumps(config, indent=2))

In [ ]:
tf.keras.utils.set_random_seed(config["seed"])  # fix R1: seed didefinisikan tapi tidak dipakai di eksperimen lama

# Hash config -- run_dir stabil untuk config yang sama (resume jalan lintas
# eksekusi notebook), tapi berubah otomatis kalau config diubah (jadi run
# baru, bukan tercampur dengan hasil config lama).
config_str = json.dumps(config, sort_keys=True)
config_hash = hashlib.sha256(config_str.encode()).hexdigest()[:8]
run_id = f"{BACKBONE}_{config_hash}"
run_dir = f"{RUNS_ROOT}/{run_id}"

print("run_id:", run_id)
print("run_dir:", run_dir)

## Data -- dari split_manifest.json, BUKAN struktur folder Kaggle asli

In [ ]:
manifest_path = f"{REPO_DIR}/results/split_manifest.json"

train_gen, val_gen, test_gen = make_generators(
    manifest_path=manifest_path,
    dataset_root=DATASET_ROOT,
    img_size=config["img_size"],
    batch_size=config["batch_size"],
    seed=config["seed"],
    augment_train=config["augment_train"],
)

class_weight = compute_class_weight(manifest_path, DATASET_ROOT) if config["use_class_weight"] else None
print("class_indices:", train_gen.class_indices)
print("class_weight:", class_weight)
print(f"train={train_gen.samples}  val={val_gen.samples}  test={test_gen.samples}")

## Training -- resume otomatis kalau run_dir sudah punya checkpoint

In [ ]:
best_model_path = run_training(
    backbone_name=BACKBONE,
    train_gen=train_gen,
    val_gen=val_gen,
    class_weight=class_weight,
    run_dir=run_dir,
    config=config,
)
print("Model terbaik tersimpan di:", best_model_path)

## Sanity check cepat di test set (evaluasi formal ada di 03_evaluate_export.ipynb)

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, classification_report

model = tf.keras.models.load_model(best_model_path)

y_prob = model.predict(test_gen, steps=len(test_gen), verbose=1).ravel()
y_true = test_gen.classes
y_pred = (y_prob >= 0.5).astype(int)

print("Accuracy (threshold 0.5, sanity check saja):", accuracy_score(y_true, y_pred))
print()
print(classification_report(y_true, y_pred, target_names=sorted(train_gen.class_indices, key=train_gen.class_indices.get), digits=4))

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

hist1 = pd.read_csv(f"{run_dir}/history_phase1.csv")
hist2 = pd.read_csv(f"{run_dir}/history_phase2.csv")
hist = pd.concat([hist1, hist2], ignore_index=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(hist["accuracy"], label="train")
axes[0].plot(hist["val_accuracy"], label="val")
axes[0].axvline(config["phase1"]["epochs"], color="gray", linestyle="--", label="mulai fase2")
axes[0].set_title(f"{BACKBONE} -- Accuracy")
axes[0].legend()

axes[1].plot(hist["loss"], label="train")
axes[1].plot(hist["val_loss"], label="val")
axes[1].axvline(config["phase1"]["epochs"], color="gray", linestyle="--")
axes[1].set_title(f"{BACKBONE} -- Loss")
axes[1].legend()

plt.tight_layout()
plt.savefig(f"{run_dir}/training_curve.png", dpi=120)
plt.show()